# DSO 576 Module 6 - Project Data Cleaning

## Project Vision and Business Question

**Project vision:** An Appliances category manager at Amazon must decide which appliance listings should be prioritized for closer review based on customer feedback.

**Business question:** Which appliance products show customer-review patterns that may indicate a need for closer listing review?

**Main comparison:** Review ratings, review text, helpful votes, and repeated product-level review activity across `parent_asin`.

## Unit of Analysis

One row represents one Amazon customer review.

The dataset does not contain a dedicated unique review ID. `parent_asin`, `asin`, and `user_id` may repeat because multiple reviews can refer to the same product or the same reviewer may write multiple reviews. Therefore, repeated identifiers are checked separately from exact duplicate rows.

## Data Source and Preservation

The project uses a 1,000-review sample from the Amazon Reviews 2023 Appliances dataset as the team's complete working dataset for this assignment.

The full Appliances dataset is substantially larger, so the team uses this 1,000-review project sample for the repository workflow. This sampling approach was discussed with and approved by the professor.

The original project sample is preserved unchanged at:

`data/raw/Appliances_sample.jsonl`

The file is loaded with `pd.read_json(..., lines=True)`. All inspection and cleaning operations are performed in memory, and the original JSONL file is not overwritten.

In [1]:
import pandas as pd

In [2]:
reviews_full = pd.read_json(
    "data/raw/Appliances_sample.jsonl",
    lines=True
)

reviews_full.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Work great,work great. use a new one every month,[],B01N0TQ0OH,B01N0TQ0OH,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2018-02-22 16:31:48.692,0,True
1,5,excellent product,Little on the thin side,[],B07DD2DMXB,B07DD37QPZ,AHWWLSPCJMALVHDDVSUGICL6RUCA,2022-10-02 21:41:03.446,0,True
2,5,Happy customer!,"Quick delivery, fixed the issue!",[],B082W3Z9YK,B082W3Z9YK,AHZIJGKEWRTAEOZ673G5B3SNXEGQ,2020-12-06 03:30:35.363,0,True
3,5,Amazing value,I wasn't sure whether these were worth it or n...,[],B078W2BJY8,B078W2BJY8,AFGUPTDFAWOHHL4LZDV27ERDNOYQ,2018-08-12 20:03:04.306,0,True
4,5,Dryer parts,Easy to install got the product expected to re...,[],B08C9LPCQV,B08C9LPCQV,AELFJFAXQERUSMTXJQ6SYFFRDWMA,2021-05-05 01:03:23.754,0,True


In [3]:
reviews_full.shape

(1000, 10)

In [4]:
reviews_full.columns

Index(['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id',
       'timestamp', 'helpful_vote', 'verified_purchase'],
      dtype='str')

In [5]:
reviews_full.dtypes

rating                        int64
title                           str
text                            str
images                       object
asin                            str
parent_asin                     str
user_id                         str
timestamp            datetime64[ms]
helpful_vote                  int64
verified_purchase              bool
dtype: object

In [6]:
cleaning_rules = pd.DataFrame({
    "rule": [
        "Keep ratings only if they are between 1 and 5.",
        "Treat blank review text as missing text.",
        "Treat blank review titles as missing titles.",
        "Remove exact duplicate review rows.",
        "Keep repeated parent_asin values because one product can have multiple reviews.",
        "Keep helpful_vote values if they are non-negative.",
        "Keep timestamps if they are valid and not in the future.",
        "Preserve identifiers such as asin, parent_asin, and user_id as text."
    ],
    "reason": [
        "Amazon ratings should use the 1-5 rating scale.",
        "Whitespace-only review text does not provide usable review content.",
        "Whitespace-only titles do not provide usable information.",
        "Exact duplicates would double-count the same observation.",
        "Repeated product IDs are expected in review-level data.",
        "Negative helpful-vote counts are not meaningful.",
        "Invalid or future review dates would be suspicious.",
        "Identifiers are labels, not quantities."
    ]
})

cleaning_rules

,rule,reason
0,Keep ratings only if they are between 1 and 5.,Amazon ratings should use the 1-5 rating scale.
1,Treat blank review text as missing text.,Whitespace-only review text does not provide u...
2,Treat blank review titles as missing titles.,Whitespace-only titles do not provide usable i...
3,Remove exact duplicate review rows.,Exact duplicates would double-count the same o...
4,Keep repeated parent_asin values because one p...,Repeated product IDs are expected in review-le...
5,Keep helpful_vote values if they are non-negat...,Negative helpful-vote counts are not meaningful.
6,Keep timestamps if they are valid and not in t...,Invalid or future review dates would be suspic...
7,"Preserve identifiers such as asin, parent_asin...","Identifiers are labels, not quantities."


In [7]:
reviews_full.isna().sum()

rating               0
title                0
text                 0
images               0
asin                 0
parent_asin          0
user_id              0
timestamp            0
helpful_vote         0
verified_purchase    0
dtype: int64

In [8]:
blank_text_count = (
    reviews_full["text"].str.strip() == ""
).sum()

blank_title_count = (
    reviews_full["title"].str.strip() == ""
).sum()

print("Blank review text:", blank_text_count)
print("Blank review titles:", blank_title_count)

Blank review text: 0
Blank review titles: 0


In [9]:
title_whitespace_changes = (
    reviews_full["title"] != reviews_full["title"].str.strip()
).sum()

text_whitespace_changes = (
    reviews_full["text"] != reviews_full["text"].str.strip()
).sum()

print("Titles changed by strip():", title_whitespace_changes)
print("Review texts changed by strip():", text_whitespace_changes)

Titles changed by strip(): 0
Review texts changed by strip(): 0


In [10]:
text_before_after = pd.DataFrame({
    "before": reviews_full["text"].head(3),
    "after": reviews_full["text"].head(3).str.strip()
})

text_before_after

,before,after
0,work great. use a new one every month,work great. use a new one every month
1,Little on the thin side,Little on the thin side
2,"Quick delivery, fixed the issue!","Quick delivery, fixed the issue!"


### Text Cleaning Decision

Leading and trailing whitespace was checked using `str.strip()`.

The before-and-after examples show no meaningful text changes were required. Review text was otherwise preserved because capitalization, punctuation, spelling, and wording are part of the original customer feedback and may be analytically meaningful.

Identifiers such as `asin`, `parent_asin`, and `user_id` were preserved as text and were not numerically transformed.

In [11]:
reviews_compare = reviews_full.copy()

reviews_compare["images"] = (
    reviews_compare["images"].astype(str)
)

exact_duplicate_count = (
    reviews_compare.duplicated().sum()
)

print(
    "Exact duplicate rows:",
    exact_duplicate_count
)

Exact duplicate rows: 0


In [12]:
print(
    "Unique parent_asin:",
    reviews_full["parent_asin"].nunique()
)

print(
    "Repeated parent_asin rows:",
    reviews_full["parent_asin"].duplicated().sum()
)

print(
    "Repeated asin rows:",
    reviews_full["asin"].duplicated().sum()
)

print(
    "Repeated user_id rows:",
    reviews_full["user_id"].duplicated().sum()
)

Unique parent_asin: 829
Repeated parent_asin rows: 171
Repeated asin rows: 130
Repeated user_id rows: 375


In [13]:
reviews_full["rating"].value_counts().sort_index()

rating
1     72
2     33
3     47
4    113
5    735
Name: count, dtype: int64

In [14]:
invalid_ratings = (
    ~reviews_full["rating"].between(1, 5)
).sum()

print("Invalid ratings:", invalid_ratings)

Invalid ratings: 0


In [15]:
negative_helpful_votes = (
    reviews_full["helpful_vote"] < 0
).sum()

print(
    "Negative helpful votes:",
    negative_helpful_votes
)

reviews_full["helpful_vote"].describe()

Negative helpful votes: 0


count    1000.000000
mean        1.229000
std         7.335146
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       185.000000
Name: helpful_vote, dtype: float64

In [16]:
print(
    "Earliest review:",
    reviews_full["timestamp"].min()
)

print(
    "Latest review:",
    reviews_full["timestamp"].max()
)

Earliest review: 2009-09-27 16:08:41
Latest review: 2023-03-14 02:44:35.407000


In [17]:
reference_date = pd.Timestamp("2026-10-04")

future_reviews = (
    reviews_full["timestamp"] > reference_date
).sum()

print(
    "Reviews after reference date:",
    future_reviews
)

Reviews after reference date: 0


In [18]:
reviews_clean = reviews_full.copy()

reviews_clean.shape

(1000, 10)

## Cleaning Result

The predefined cleaning rules were applied to all 1,000 project reviews.

No rows required removal or correction because the dataset contained:

- no missing values,
- no blank review text or titles,
- no exact duplicate rows,
- no invalid ratings,
- no negative helpful-vote counts, and
- no invalid or future timestamps.

Therefore, all 1,000 rows were retained in the cleaned dataset.

In [19]:
print(
    "Original rows:",
    len(reviews_full)
)

print(
    "Cleaned rows:",
    len(reviews_clean)
)

print(
    "Rows removed:",
    len(reviews_full) - len(reviews_clean)
)

Original rows: 1000
Cleaned rows: 1000
Rows removed: 0


In [20]:
rating_count = reviews_clean["rating"].count()
average_rating = reviews_clean["rating"].mean()

verified_count = (
    reviews_clean["verified_purchase"].sum()
)

total_reviews = len(reviews_clean)

print(
    "Rating denominator:",
    rating_count
)

print(
    "Average rating:",
    average_rating
)

print(
    "Verified purchases:",
    verified_count
)

print(
    "Total reviews:",
    total_reviews
)

Rating denominator: 1000
Average rating: 4.406
Verified purchases: 898
Total reviews: 1000


In [21]:
reviews_clean[
    "verified_purchase"
].value_counts()

verified_purchase
True     898
False    102
Name: count, dtype: int64

## Calculation and Denominator Checks

The dataset contains no missing ratings, so the average rating uses all 1,000 reviews as its denominator.

The verified-purchase count also uses all 1,000 reviews because `verified_purchase` has no missing values.

There are no missing `parent_asin` values in the project dataset, so no records need to be excluded from product-level grouping because of missing group keys.

If missing values were present in a metric column, they would not be treated as zero unless there were a clear business reason to do so.

In [22]:
record_1 = reviews_full[
    reviews_full["rating"] == 1
].head(1)

record_2 = reviews_full[
    reviews_full["rating"] == 5
].head(1)

record_3 = reviews_full[
    reviews_full["verified_purchase"] == False
].head(1)

record_4 = reviews_full[
    reviews_full["helpful_vote"] ==
    reviews_full["helpful_vote"].max()
].head(1)

repeated_parent = (
    reviews_full["parent_asin"]
    .value_counts()
    .index[0]
)

record_5 = reviews_full[
    reviews_full["parent_asin"] ==
    repeated_parent
].head(1)

In [23]:
record_checks = pd.concat([
    record_1,
    record_2,
    record_3,
    record_4,
    record_5
])

record_checks[
    [
        "rating",
        "title",
        "parent_asin",
        "timestamp",
        "helpful_vote",
        "verified_purchase"
    ]
]

,rating,title,parent_asin,timestamp,helpful_vote,verified_purchase
17,1,horrid,B07VN6QRQL,2017-07-15 03:42:32.427,0,True
0,5,Work great,B01N0TQ0OH,2018-02-22 16:31:48.692,0,True
59,5,Fits well.,B0C2Z4QHXG,2022-11-15 11:20:35.734,0,False
904,5,Little workhorse washing machine,B07LCSK4JY,2020-09-07 19:32:46.501,185,True
60,5,"Works great, easy to clean",B07RNJY499,2018-02-06 23:44:15.155,0,True


In [24]:
record_check_plan = pd.DataFrame({
    "record_type": [
        "1-star review",
        "5-star review",
        "Unverified purchase",
        "Highest helpful_vote",
        "Repeated parent_asin"
    ],
    "expected_result": [
        "Keep unchanged",
        "Keep unchanged",
        "Keep unchanged",
        "Keep unchanged",
        "Keep unchanged"
    ],
    "reason": [
        "A rating of 1 is valid on the Amazon 1-5 rating scale.",
        "A rating of 5 is valid on the Amazon 1-5 rating scale.",
        "An unverified purchase is still a valid review record.",
        "A high helpful_vote value may be unusual but is not invalid.",
        "Repeated parent_asin values are expected because a product can have multiple reviews."
    ]
})

record_check_plan

,record_type,expected_result,reason
0,1-star review,Keep unchanged,A rating of 1 is valid on the Amazon 1-5 ratin...
1,5-star review,Keep unchanged,A rating of 5 is valid on the Amazon 1-5 ratin...
2,Unverified purchase,Keep unchanged,An unverified purchase is still a valid review...
3,Highest helpful_vote,Keep unchanged,A high helpful_vote value may be unusual but i...
4,Repeated parent_asin,Keep unchanged,Repeated parent_asin values are expected becau...


In [25]:
record_checks_clean = reviews_clean.loc[
    record_checks.index
]

record_checks_clean[
    [
        "rating",
        "title",
        "parent_asin",
        "timestamp",
        "helpful_vote",
        "verified_purchase"
    ]
]

,rating,title,parent_asin,timestamp,helpful_vote,verified_purchase
17,1,horrid,B07VN6QRQL,2017-07-15 03:42:32.427,0,True
0,5,Work great,B01N0TQ0OH,2018-02-22 16:31:48.692,0,True
59,5,Fits well.,B0C2Z4QHXG,2022-11-15 11:20:35.734,0,False
904,5,Little workhorse washing machine,B07LCSK4JY,2020-09-07 19:32:46.501,185,True
60,5,"Works great, easy to clean",B07RNJY499,2018-02-06 23:44:15.155,0,True


In [26]:
record_check_results = []

for i, index in enumerate(record_checks.index):
    original = reviews_full.loc[index]
    actual = reviews_clean.loc[index]

    match = original.equals(actual)

    record_check_results.append({
        "record_type": record_check_plan.loc[i, "record_type"],
        "original_rating": original["rating"],
        "original_parent_asin": original["parent_asin"],
        "expected_result": record_check_plan.loc[i, "expected_result"],
        "actual_result": "Kept unchanged" if match else "Changed",
        "matches_expected": match
    })

record_check_results = pd.DataFrame(record_check_results)

record_check_results

,record_type,original_rating,original_parent_asin,expected_result,actual_result,matches_expected
0,1-star review,1,B07VN6QRQL,Keep unchanged,Kept unchanged,True
1,5-star review,5,B01N0TQ0OH,Keep unchanged,Kept unchanged,True
2,Unverified purchase,5,B0C2Z4QHXG,Keep unchanged,Kept unchanged,True
3,Highest helpful_vote,5,B07LCSK4JY,Keep unchanged,Kept unchanged,True
4,Repeated parent_asin,5,B07RNJY499,Keep unchanged,Kept unchanged,True


In [27]:
reconciliation = pd.DataFrame({
    "stage": [
        "Original sample",
        "After cleaning"
    ],
    "rows": [
        len(reviews_full),
        len(reviews_clean)
    ]
})

reconciliation

,stage,rows
0,Original sample,1000
1,After cleaning,1000


In [28]:
difficult_cases = pd.concat([
    reviews_clean[
        reviews_clean["rating"] == 1
    ].head(1),

    reviews_clean[
        reviews_clean["rating"] == 5
    ].head(1),

    reviews_clean[
        reviews_clean["verified_purchase"] == False
    ].head(1),

    reviews_clean[
        reviews_clean["helpful_vote"] ==
        reviews_clean["helpful_vote"].max()
    ].head(1),

    reviews_clean[
        reviews_clean["parent_asin"] ==
        repeated_parent
    ].head(1)
])

In [29]:
normal_cases = reviews_clean.drop(
    index=difficult_cases.index,
    errors="ignore"
).head(45)

In [30]:
submission_sample = pd.concat(
    [
        difficult_cases,
        normal_cases
    ],
    ignore_index=True
)

submission_sample.shape

(50, 10)

## Submission Sample Selection

The submitted cleaned sample contains 50 rows.

Five rows were selected to represent notable or potentially difficult cases:

1. a 1-star review,
2. a 5-star review,
3. an unverified purchase,
4. the review with the highest helpful-vote count, and
5. a review from a repeated `parent_asin`.

The remaining 45 rows were selected from the cleaned dataset after excluding those five records.

This sample is intended to provide the instructor with both representative and edge-case records while staying within the Module 6 limit of up to 50 cleaned rows.

In [31]:
submission_sample.to_csv(
    "cleaned_sample.csv",
    index=False
)

In [32]:
print(
    "Original shape:",
    reviews_full.shape
)

print(
    "Cleaned shape:",
    reviews_clean.shape
)

print(
    "Missing values:",
    reviews_clean.isna().sum().sum()
)

print(
    "Blank text:",
    (
        reviews_clean["text"].str.strip() == ""
    ).sum()
)

print(
    "Blank titles:",
    (
        reviews_clean["title"].str.strip() == ""
    ).sum()
)

clean_compare = reviews_clean.copy()

clean_compare["images"] = (
    clean_compare["images"].astype(str)
)

print(
    "Exact duplicates:",
    clean_compare.duplicated().sum()
)

print(
    "Invalid ratings:",
    (
        ~reviews_clean["rating"].between(1, 5)
    ).sum()
)

print(
    "Negative helpful votes:",
    (
        reviews_clean["helpful_vote"] < 0
    ).sum()
)

print(
    "Submission sample:",
    submission_sample.shape
)

Original shape: (1000, 10)
Cleaned shape: (1000, 10)
Missing values: 0
Blank text: 0
Blank titles: 0
Exact duplicates: 0
Invalid ratings: 0
Negative helpful votes: 0
Submission sample: (50, 10)


## Reproducibility

After the notebook was finalized, the kernel was restarted and all cells were run from top to bottom using the listed project dependencies.

The workflow reproduced the same validation results and recreated the 50-row `cleaned_sample.csv`.

Expected final results:

- Original dataset: 1,000 rows
- Cleaned dataset: 1,000 rows
- Missing values: 0
- Blank review text: 0
- Blank review titles: 0
- Exact duplicate rows: 0
- Invalid ratings: 0
- Negative helpful votes: 0
- Submission sample: 50 rows

## Remaining Limitations

1. The 1,000-review sample is only a subset of the full Amazon Appliances review dataset, so results should not be interpreted as representing all Appliances reviews.

2. Review text may contain subjective opinions, spelling differences, product-use issues, or delivery-related comments that are not directly related to listing quality.

3. This sample contains review data only. Product metadata such as features, descriptions, and prices is not included, so this cleaning workflow cannot directly compare customer reviews with listing information.

4. Repeated `parent_asin` values are expected because multiple reviews may refer to the same parent product. These records were retained rather than treated as duplicate rows.

In [33]:
cleaning_decision_log = pd.DataFrame({
    "rule": [
        "Keep ratings only if they are between 1 and 5.",
        "Treat blank review text as missing content.",
        "Treat blank review titles as missing content.",
        "Remove exact duplicate rows if found.",
        "Keep repeated product and user identifiers.",
        "Keep helpful_vote only if non-negative.",
        "Flag invalid or future timestamps.",
        "Preserve original review wording and identifiers."
    ],
    "reason": [
        "Amazon ratings use a 1-5 scale.",
        "Blank text cannot support review-text analysis.",
        "Blank titles contain no review-title information.",
        "Exact duplicates could double-count the same observation.",
        "Multiple reviews per product or reviewer are expected.",
        "Negative helpful-vote counts are not meaningful.",
        "Invalid or future dates would be suspicious.",
        "Raw customer wording and identifier formatting may be meaningful."
    ],
    "check": [
        "0 invalid ratings found.",
        "0 blank review texts found.",
        "0 blank review titles found.",
        "0 exact duplicate rows found.",
        "Repeated IDs were inspected separately and retained.",
        "0 negative helpful-vote values found.",
        "0 future dates found.",
        "No destructive text normalization was applied."
    ]
})

cleaning_decision_log

,rule,reason,check
0,Keep ratings only if they are between 1 and 5.,Amazon ratings use a 1-5 scale.,0 invalid ratings found.
1,Treat blank review text as missing content.,Blank text cannot support review-text analysis.,0 blank review texts found.
2,Treat blank review titles as missing content.,Blank titles contain no review-title information.,0 blank review titles found.
3,Remove exact duplicate rows if found.,Exact duplicates could double-count the same o...,0 exact duplicate rows found.
4,Keep repeated product and user identifiers.,Multiple reviews per product or reviewer are e...,Repeated IDs were inspected separately and ret...
5,Keep helpful_vote only if non-negative.,Negative helpful-vote counts are not meaningful.,0 negative helpful-vote values found.
6,Flag invalid or future timestamps.,Invalid or future dates would be suspicious.,0 future dates found.
7,Preserve original review wording and identifiers.,Raw customer wording and identifier formatting...,No destructive text normalization was applied.
